# Trabalho 1 — Aquisição de Dados (Cinema)

Coleta TMDB (API) + OMDb (API) + Letterboxd (scraping), integração e limpeza.

**Antes de rodar:** preencha `TMDB_API_KEY` e `OMDB_API_KEY` em `.env` (veja o README).

## 0. Setup e utilitários

In [ ]:
from __future__ import annotations

import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tqdm.auto import tqdm

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT / "trabalho01" / "requirements.txt").exists():
    ROOT = ROOT / "trabalho01"

DIR_BRUTOS = ROOT / "dados_brutos"
DIR_TRATADOS = ROOT / "dados_tratados"
DIR_DOCS = ROOT / "docs"
LOG_PATH = DIR_DOCS / "proveniencia.jsonl"

for d in (DIR_BRUTOS, DIR_TRATADOS, DIR_DOCS):
    d.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")
TMDB_API_KEY = os.getenv("TMDB_API_KEY", "").strip()
OMDB_API_KEY = os.getenv("OMDB_API_KEY", "").strip()

HEADERS = {
    "User-Agent": (
        "UFAM-CD-Trabalho1-Cinema/1.0 "
        "(+academic; contato via repositorio do projeto)"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

print("ROOT:", ROOT)
print("TMDB_API_KEY definida:", bool(TMDB_API_KEY))
print("OMDB_API_KEY definida:", bool(OMDB_API_KEY))
if not TMDB_API_KEY or not OMDB_API_KEY:
    print("AVISO: preencha trabalho01/.env com TMDB_API_KEY e OMDB_API_KEY antes da coleta.")

In [ ]:
def log_proveniencia(
    fonte: str,
    url: str,
    metodo: str,
    status: int | None,
    params: dict | None = None,
    observacao: str = "",
) -> None:
    """Append de uma linha JSON no registro de proveniência."""
    registro = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "fonte": fonte,
        "url": url,
        "metodo": metodo,
        "params": params or {},
        "status": status,
        "observacao": observacao,
    }
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")


def get_com_retry(
    url: str,
    *,
    params: dict | None = None,
    headers: dict | None = None,
    timeout: int = 30,
    max_tentativas: int = 3,
    sleep_base: float = 1.0,
) -> requests.Response:
    """GET com retentativas para 429/5xx."""
    ultimo_erro: Exception | None = None
    for tentativa in range(1, max_tentativas + 1):
        try:
            resp = requests.get(
                url,
                params=params,
                headers=headers or HEADERS,
                timeout=timeout,
            )
            if resp.status_code in {429, 500, 502, 503, 504}:
                time.sleep(sleep_base * tentativa)
                continue
            return resp
        except requests.RequestException as exc:
            ultimo_erro = exc
            time.sleep(sleep_base * tentativa)
    if ultimo_erro:
        raise ultimo_erro
    raise RuntimeError(f"Falha ao obter {url}")


print("Utilitários carregados. Log:", LOG_PATH)

## 1. Teste das API keys (rode após preencher o `.env`)

In [ ]:
if TMDB_API_KEY:
    r_tmdb = get_com_retry(
        "https://api.themoviedb.org/3/movie/550",
        params={"api_key": TMDB_API_KEY},
    )
    print("TMDB:", r_tmdb.status_code, r_tmdb.json().get("title"))
    log_proveniencia(
        "TMDB",
        r_tmdb.url.replace(TMDB_API_KEY, "***"),
        "API GET /movie/550",
        r_tmdb.status_code,
        {"movie_id": 550},
        "teste de chave",
    )
else:
    print("TMDB: chave ausente")

if OMDB_API_KEY:
    r_omdb = get_com_retry(
        "https://www.omdbapi.com/",
        params={"i": "tt0137523", "apikey": OMDB_API_KEY},
    )
    data = r_omdb.json()
    print("OMDb:", r_omdb.status_code, data.get("Title"), data.get("imdbRating"), data.get("Metascore"))
    log_proveniencia(
        "OMDb",
        "https://www.omdbapi.com/?i=tt0137523&apikey=***",
        "API GET",
        r_omdb.status_code,
        {"i": "tt0137523"},
        "teste de chave",
    )
else:
    print("OMDb: chave ausente")

## 2. Coleta TMDB (API)

- `discover/movie` com `sort_by=revenue.desc`, páginas 1–50 (~1000 IDs)
- Detalhes em `/movie/{id}` para cada filme
- Salva **todos** os registros em `dados_brutos/tmdb_raw.csv` (sem filtrar `imdb_id`; isso fica para a OMDb)
- Proveniência em `docs/proveniencia.jsonl` (API key mascarada)
- Esta célula **substitui** o CSV bruto completo (expansão 400→1000)

In [ ]:
assert TMDB_API_KEY, "TMDB_API_KEY ausente no .env"

TMDB_DISCOVER_URL = "https://api.themoviedb.org/3/discover/movie"
TMDB_DETAIL_URL = "https://api.themoviedb.org/3/movie/{movie_id}"
TMDB_PAGES = range(1, 51)  # 50 páginas × 20 resultados = ~1000
TMDB_SLEEP = 0.25
TMDB_RAW_PATH = DIR_BRUTOS / "tmdb_raw.csv"

# --- Discover: IDs únicos ---
tmdb_ids: list[int] = []
seen: set[int] = set()

for page in tqdm(list(TMDB_PAGES), desc="TMDB discover"):
    params = {
        "api_key": TMDB_API_KEY,
        "sort_by": "revenue.desc",
        "page": page,
        "include_adult": "false",
    }
    resp = get_com_retry(TMDB_DISCOVER_URL, params=params)
    log_proveniencia(
        "TMDB",
        f"{TMDB_DISCOVER_URL}?sort_by=revenue.desc&page={page}&api_key=***",
        "API GET /discover/movie",
        resp.status_code,
        {"sort_by": "revenue.desc", "page": page, "include_adult": False},
    )
    resp.raise_for_status()
    for item in resp.json().get("results", []):
        mid = item.get("id")
        if mid is not None and mid not in seen:
            seen.add(mid)
            tmdb_ids.append(mid)
    time.sleep(TMDB_SLEEP)

print(f"IDs únicos coletados no discover: {len(tmdb_ids)}")

# --- Details: todos os filmes (com ou sem imdb_id) ---
rows: list[dict] = []

for mid in tqdm(tmdb_ids, desc="TMDB details"):
    url = TMDB_DETAIL_URL.format(movie_id=mid)
    resp = get_com_retry(url, params={"api_key": TMDB_API_KEY})
    log_proveniencia(
        "TMDB",
        f"{url}?api_key=***",
        "API GET /movie/{id}",
        resp.status_code,
        {"movie_id": mid},
    )
    if resp.status_code != 200:
        time.sleep(TMDB_SLEEP)
        continue
    data = resp.json()
    genres = "|".join(g.get("name", "") for g in data.get("genres") or [])
    rows.append(
        {
            "tmdb_id": data.get("id"),
            "imdb_id": data.get("imdb_id") or "",
            "title": data.get("title"),
            "release_date": data.get("release_date"),
            "budget": data.get("budget"),
            "revenue": data.get("revenue"),
            "genres": genres,
            "runtime": data.get("runtime"),
            "original_language": data.get("original_language"),
            "tmdb_vote_average": data.get("vote_average"),
            "tmdb_vote_count": data.get("vote_count"),
        }
    )
    time.sleep(TMDB_SLEEP)

df_tmdb = pd.DataFrame(rows)
df_tmdb.to_csv(TMDB_RAW_PATH, index=False, encoding="utf-8")
print(f"Salvo: {TMDB_RAW_PATH}")
print(f"Linhas: {len(df_tmdb)} | Colunas: {list(df_tmdb.columns)}")
print(f"Com imdb_id: {(df_tmdb['imdb_id'].astype(str).str.len() > 0).sum()}")
print(f"Sem imdb_id: {(df_tmdb['imdb_id'].astype(str).str.len() == 0).sum()}")
df_tmdb.head()


## Próximos passos

1. Coleta OMDb → `dados_brutos/omdb_raw.csv` (apenas linhas com `imdb_id`)
2. Scraping Letterboxd → `dados_brutos/letterboxd_raw.csv`
3. Join + limpeza → `dados_tratados/base_tratada.parquet`
4. Dataset Card + proveniência completa
